In [1]:
import os

In [2]:
getPath = os.getcwd()
file:str = 'input2.txt'
print(getPath)

/home/mariano/PRACTICA2/Programas


In [3]:
newPath = os.path.join(getPath, 'dataset', file)

In [4]:
print(newPath)

/home/mariano/PRACTICA2/Programas/dataset/input2.txt


In [6]:
with open(newPath, 'r', encoding='utf-8') as f:
    text = f.read()
print(len(text))

147758


In [7]:
print(text[:100])

<|inicio|>
Era de aspecto venerable y viejo;
de verde, azul y plata era el vestido,
robusto al parec


In [8]:
chars = sorted(list(set(text)))
vocabSize = len(chars)
print(''.join(chars))
print(f"Tamaño del vocabulario: {vocabSize}")


 ,.;<>ABCDEFGHIJLMNOPQRSTUVXYZabcdefghijlmnopqrstuvxyz|ÁÉÍÑÓÚÜáéíñóúü
Tamaño del vocabulario: 70


In [9]:
#Tokenizador de 1 a 1:Caracteres
stoi = {ch:i for i,ch in enumerate(chars)} #String to Integer
itos = {i:ch for i,ch in enumerate(chars)} #Integer to String
encoder = lambda s: [stoi[c] for c in s]
decoder = lambda l: ''.join([itos[i] for i in l])
print(encoder("Damian"))
print(decoder([10, 31, 42, 39, 31, 43]))


[10, 31, 42, 39, 31, 43]
Damian


In [10]:
class BabyTokenizer:
    def __init__(self):
        self.merges = {} 
        self.vocab = {i: bytes([i]) for i in range(256)}

    def _get_stats(self, ids):
        counts = {}
        for pair in zip(ids, ids[1:]):
            counts[pair] = counts.get(pair, 0) + 1
        return counts

    def _merge(self, ids, pair, idx):
        newids = []
        i = 0
        while i < len(ids):
            if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
                newids.append(idx)
                i += 2
            else:
                newids.append(ids[i])
                i += 1
        return newids

    def train(self, text, vocab_size, verbose=False):
        num_merges = vocab_size - 256
        text_bytes = text.encode("utf-8")
        ids = list(text_bytes)

        for i in range(num_merges):
            stats = self._get_stats(ids)
            if not stats: break
            
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = self._merge(ids, pair, idx)
            self.merges[pair] = idx
            self.vocab[idx] = self.vocab[pair[0]] + self.vocab[pair[1]]
            
            if verbose:
                print(f"Merge {i+1}/{num_merges}: {pair} -> {idx} ({self.vocab[idx].decode('utf-8', errors='replace')})")

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = self._get_stats(ids)
            pair = min(stats.keys(), key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            idx = self.merges[pair]
            ids = self._merge(ids, pair, idx)
        return ids

    def decode(self, ids):
        text_bytes = b"".join(self.vocab[idx] for idx in ids)
        return text_bytes.decode("utf-8", errors="replace")

In [11]:
tokenizer = BabyTokenizer()

training_data = text

tokenizer.train(training_data, vocab_size=2048, verbose=True)
print(len(tokenizer.vocab))


Merge 1/1792: (101, 32) -> 256 (e )
Merge 2/1792: (97, 32) -> 257 (a )
Merge 3/1792: (111, 32) -> 258 (o )
Merge 4/1792: (115, 32) -> 259 (s )
Merge 5/1792: (101, 110) -> 260 (en)
Merge 6/1792: (108, 32) -> 261 (l )
Merge 7/1792: (101, 115) -> 262 (es)
Merge 8/1792: (44, 32) -> 263 (, )
Merge 9/1792: (113, 117) -> 264 (qu)
Merge 10/1792: (101, 114) -> 265 (er)
Merge 11/1792: (97, 110) -> 266 (an)
Merge 12/1792: (44, 10) -> 267 (,
)
Merge 13/1792: (121, 32) -> 268 (y )
Merge 14/1792: (100, 256) -> 269 (de )
Merge 15/1792: (105, 110) -> 270 (in)
Merge 16/1792: (111, 114) -> 271 (or)
Merge 17/1792: (97, 114) -> 272 (ar)
Merge 18/1792: (264, 256) -> 273 (que )
Merge 19/1792: (111, 110) -> 274 (on)
Merge 20/1792: (101, 261) -> 275 (el )
Merge 21/1792: (10, 10) -> 276 (

)
Merge 22/1792: (111, 259) -> 277 (os )
Merge 23/1792: (99, 105) -> 278 (ci)
Merge 24/1792: (108, 257) -> 279 (la )
Merge 25/1792: (260, 32) -> 280 (en )
Merge 26/1792: (97, 259) -> 281 (as )
Merge 27/1792: (97, 100) -> 282

In [12]:
test_phrase = "Era de aspecto venerable y viejo; de verde, azul y plata era el vestido, robusto al parec"
tokens = tokenizer.encode(test_phrase)

print(f"\nFrase: '{test_phrase}'")
print(f"Longitud original (bytes): {len(test_phrase.encode('utf-8'))}")
print(f"Longitud comprimida (tokens): {len(tokens)}")
print(f"Tokens: {tokens}")
print(f"Decodificado: {tokenizer.decode(tokens)}")


Frase: 'Era de aspecto venerable y viejo; de verde, azul y plata era el vestido, robusto al parec'
Longitud original (bytes): 89
Longitud comprimida (tokens): 33
Tokens: [1731, 422, 285, 112, 413, 930, 546, 265, 991, 268, 310, 477, 1211, 269, 386, 100, 367, 398, 117, 261, 268, 1928, 257, 265, 485, 1286, 314, 114, 111, 562, 320, 894, 632]
Decodificado: Era de aspecto venerable y viejo; de verde, azul y plata era el vestido, robusto al parec


In [13]:
import torch

In [14]:
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
print(data[:100])

tensor([ 333, 1731,  422,  285,  112,  413,  930,  546,  265,  991,  268,  310,
         477,  493,  269,  386,  100,  367,  398,  117,  261,  268, 1928,  257,
         265,  485, 1286,  305,  114,  111,  562,  320,  894, 1105, 1216, 1287,
         306,  106,  417,  555,  633,  634, 1732, 1366,  100,  260,  368,  317,
        1025, 1733,  311,  504,  275, 1153,  649,  279,  115,  426,  309,  666,
         877,  485, 1455,  397,  496,  931,  414,  435,   65,  429, 1366,  698,
         257,  423, 1055,  321, 1734,  550,  426,  676,  857,  340, 1367,  115,
         436,  115,  302,  108,  299,  296,  303,  486,  305,  627,  122,  103,
         424,  257,  713,  257])
